In [1]:
from langgraph.graph import StateGraph , START , END
from langchain_google_genai import GoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [3]:
load_dotenv()  # Load environment variables from .env file
model = GoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.2)

In [4]:
class BlogPostState(TypedDict):
    topic: str
    outline: str
    draft: str
    final_post: str

In [ ]:
def generate_outline(state: BlogPostState) -> BlogPostState:
    prompt = f"Generate an outline for a blog post about: {state['topic']}"
    outline = model.invoke(prompt)
    state['outline'] = outline
    return state

In [10]:
def generate_draft(state: BlogPostState) -> BlogPostState:
    prompt = f"Write a draft for the blog post based on this outline: {state['outline']}"
    draft = model.invoke(prompt)
    state['draft'] = draft
    return state

In [11]:
def finalize_post(state: BlogPostState) -> BlogPostState:
    prompt = f"Finalize the blog post based on this draft: {state['draft']}"
    final_post = model.invoke(prompt)
    state['final_post'] = final_post
    return state

In [8]:
graph = StateGraph(BlogPostState)

graph.add_node('generate_outline', generate_outline)
graph.add_node('generate_draft', generate_draft)
graph.add_node('finalize_post', finalize_post)

graph.add_edge(START, 'generate_outline')
graph.add_edge('generate_outline', 'generate_draft')
graph.add_edge('generate_draft', 'finalize_post')
graph.add_edge('finalize_post', END)

workflow = graph.compile()

In [12]:
workflow.invoke({'topic': 'The Future of AI', 'outline': '', 'draft': '', 'final_post': ''})

ValueError: Argument 'prompts' is expected to be of type list[str], received argument of type <class 'str'>.